# Verify PhotoRing Simulator defaults against exorings

This notebook uses the copied reference package in `dev/exorings` to verify the default transit parameters used by `apps/photoring-simulator`.

The app defaults are passed explicitly. The package's own historical defaults are not used.

In [1]:
from pathlib import Path
import sys
import math
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "dev":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "dev"))

from exorings import ExoringsBasicParams, compute_exorings_basic, forward_observables

print("Reference package:", (REPO_ROOT / "dev" / "exorings").resolve())

Reference package: /Users/jzuluaga/dev/seap-udea.github.io/dev/exorings


In [2]:
# Exact defaults from PhotoRingSimulator.tsx, converted to the exorings API.
APP_DEFAULTS = {
    "rhotrue": 1.408,
    "P": 365.25,
    "b": 0.25,
    "p": 0.084,
    "fi": 1.58,
    "fe": 2.35,
    "tau": 1.0,
    "theta": 25.0,
    "ir": 55.0,
}
ALPHA = math.exp(-APP_DEFAULTS["tau"])
params = ExoringsBasicParams(**APP_DEFAULTS)
reference = compute_exorings_basic(params)
forward_kipping = forward_observables(
    rhotrue_gcc=APP_DEFAULTS["rhotrue"],
    P_days=APP_DEFAULTS["P"],
    b=APP_DEFAULTS["b"],
    p=APP_DEFAULTS["p"],
    fi=APP_DEFAULTS["fi"],
    fe=APP_DEFAULTS["fe"],
    alpha=ALPHA,
    theta_deg=APP_DEFAULTS["theta"],
    ir_deg=APP_DEFAULTS["ir"],
    bobs_method="kipping",
)
forward_mallen = forward_observables(
    rhotrue_gcc=APP_DEFAULTS["rhotrue"],
    P_days=APP_DEFAULTS["P"],
    b=APP_DEFAULTS["b"],
    p=APP_DEFAULTS["p"],
    fi=APP_DEFAULTS["fi"],
    fe=APP_DEFAULTS["fe"],
    alpha=ALPHA,
    theta_deg=APP_DEFAULTS["theta"],
    ir_deg=APP_DEFAULTS["ir"],
    bobs_method="mallen",
)

assert forward_kipping is not None
assert forward_mallen is not None
assert math.isclose(-math.log(ALPHA), 1.0, rel_tol=0.0, abs_tol=1e-14)
print("alpha =", ALPHA, "and tau =", -math.log(ALPHA))

alpha = 0.36787944117144233 and tau = 1.0


In [3]:
values = {
    "a_over_Rstar": reference.a,
    "beta": reference.beta,
    "depth_delta": reference.delta,
    "equivalent_radius": reference.pobs,
    "contacts_x": [reference.x1, reference.x2, reference.x3, reference.x4],
    "T14_hours": reference.T14,
    "T23_hours": reference.T23,
    "aobs_over_Rstar": forward_kipping["aobs"],
    "bobs_kipping_app": forward_kipping["bobs"],
    "bobs_mallen_reference": forward_mallen["bobs"],
    "rhoobs_gcc": reference.rhoobs,
    "rhoobs_over_rhotrue": reference.PR,
    "log10_PR": reference.logPR,
}
for name, value in values.items():
    print(f"{name}: {value}")

a_over_Rstar: 214.93870500277973
beta: 0.8250835409018498
depth_delta: 0.01691691202569003
equivalent_radius: 0.13006502998765668
contacts_x: [-1.1414600318549009, -0.7930098820617327, 0.7650080167444258, 1.1687395830218195]
T14_hours: 14.995644397347059
T23_hours: 10.113081558371782
aobs_over_Rstar: 181.77142125704745
bobs_kipping_app: 0.5681210786234157
bobs_mallen_reference: 0.8080360907628269
rhoobs_gcc: 0.8515998569546841
rhoobs_over_rhotrue: 0.6048294438598609
log10_PR: -0.21836707498931118


In [4]:
# The app uses the Kipping inversion. The OO reference result exposes the Mallen inversion.
checks = {
    "delta": (reference.delta, forward_kipping["delta"]),
    "T14": (reference.T14, forward_kipping["T14"]),
    "T23": (reference.T23, forward_kipping["T23"]),
    "rhoobs": (reference.rhoobs, forward_kipping["rhoobs"]),
    "aobs": (reference.aobs, forward_kipping["aobs"]),
    "pobs": (reference.pobs, forward_kipping["pobs"]),
}
for name, (basic_value, forward_value) in checks.items():
    difference = abs(basic_value - forward_value)
    print(f"{name}: difference = {difference:.3e}")
    assert math.isclose(basic_value, forward_value, rel_tol=1e-10, abs_tol=1e-10)

assert math.isclose(forward_kipping["bobs"], 0.5681210786234157, rel_tol=1e-12)
print(f"b_obs (Kipping, app): {forward_kipping['bobs']:.12f}")
print(f"b_obs (Mallen, alternate): {forward_mallen['bobs']:.12f}")

delta: difference = 0.000e+00
T14: difference = 0.000e+00
T23: difference = 0.000e+00
rhoobs: difference = 0.000e+00
aobs: difference = 0.000e+00
pobs: difference = 0.000e+00
b_obs (Kipping, app): 0.568121078623
b_obs (Mallen, alternate): 0.808036090763


In [5]:
# Check the app's geometry formulas against the package contact positions.
DEG = math.pi / 180.0
cos_i = math.cos(APP_DEFAULTS["ir"] * DEG)
theta = APP_DEFAULTS["theta"] * DEG
outer = APP_DEFAULTS["fe"] * APP_DEFAULTS["p"]
x0 = math.sqrt(1.0 - APP_DEFAULTS["b"] ** 2)
h_right = math.sqrt((outer * (x0 * math.cos(theta) + APP_DEFAULTS["b"] * math.sin(theta))) ** 2 + (outer * cos_i * (APP_DEFAULTS["b"] * math.cos(theta) - x0 * math.sin(theta))) ** 2)
h_left = math.sqrt((outer * (-x0 * math.cos(theta) + APP_DEFAULTS["b"] * math.sin(theta))) ** 2 + (outer * cos_i * (APP_DEFAULTS["b"] * math.cos(theta) + x0 * math.sin(theta))) ** 2)
p = APP_DEFAULTS["p"]
b = APP_DEFAULTS["b"]
ring_contacts = [
    -math.sqrt(max(0.0, (1.0 + h_left) ** 2 - b ** 2)),
    -math.sqrt(max(0.0, (1.0 - h_left) ** 2 - b ** 2)),
    math.sqrt(max(0.0, (1.0 - h_right) ** 2 - b ** 2)),
    math.sqrt(max(0.0, (1.0 + h_right) ** 2 - b ** 2)),
]
planet_contacts = [
    -math.sqrt(max(0.0, (1.0 + p) ** 2 - b ** 2)),
    -math.sqrt(max(0.0, (1.0 - p) ** 2 - b ** 2)),
    math.sqrt(max(0.0, (1.0 - p) ** 2 - b ** 2)),
    math.sqrt(max(0.0, (1.0 + p) ** 2 - b ** 2)),
]
app_contacts = [
    min(ring_contacts[0], planet_contacts[0]),
    max(ring_contacts[1], planet_contacts[1]),
    min(ring_contacts[2], planet_contacts[2]),
    max(ring_contacts[3], planet_contacts[3]),
]
print("app contacts:      ", app_contacts)
print("package contacts:  ", [reference.x1, reference.x2, reference.x3, reference.x4])
assert np.allclose(app_contacts, [reference.x1, reference.x2, reference.x3, reference.x4], rtol=0.0, atol=1e-12)

app contacts:       [-1.1414600318549009, -0.7930098820617327, 0.7650080167444258, 1.1687395830218195]
package contacts:   [-1.1414600318549009, -0.7930098820617327, 0.7650080167444258, 1.1687395830218195]


In [6]:
print("PASS: alpha/tau mapping, both exorings APIs, app contact geometry, and defaults are consistent.")
print(f"The app's b_obs uses Kipping: {forward_kipping['bobs']:.6f}; Mallen gives {forward_mallen['bobs']:.6f} as an alternate convention.")
print("Note: the app's pixel-grid occultor area is a numerical approximation; compare its displayed depth to", round(reference.delta, 8), "with the expected small grid discretization difference.")

PASS: alpha/tau mapping, both exorings APIs, app contact geometry, and defaults are consistent.
The app's b_obs uses Kipping: 0.568121; Mallen gives 0.808036 as an alternate convention.
Note: the app's pixel-grid occultor area is a numerical approximation; compare its displayed depth to 0.01691691 with the expected small grid discretization difference.
